# Введение и постановка задачи

Сервисы доставки еды уже давно перестали быть просто курьерами, которые привозят заказ. Индустрия e-grocery стремительно идет к аккумулированию и использованию больших данных, чтобы знать о своих пользователях больше и предоставлять более качественные и персонализированные услуги. Одним из шагов к такой персонализации может быть разработка модели, которая понимает привычки и нужды пользователя, и, к примеру, может угадать, что и когда пользователь захочет заказать в следующий раз.

Такая модель, будучи разработанной, может принести значительную ценность для клиента - сэкономить время при сборке корзины, помочь ничего не забыть в заказе, убрать необходимость планировать закупки и следить за заканчивающимися запасами продуктов.

В данном соревновании участникам предлагается решить задачу предсказания следующего заказа пользователя (безотносительно конкретного момента времени, когда этот заказ произойдет). Заказ пользователя состоит из списка уникальных категорий товаров, вне зависимости от того, сколько продуктов каждой категории он взял.


# Описание набора данных

В качестве тренировочных данных представляется датасет с историей заказов 20000 пользователей вплоть до даты отсечки, которая разделяет тренировочные и тестовые данные по времени.

train.csv:

user_id - уникальный id пользователя.

order_completed_at - дата заказа.

cart - список уникальных категорий (category_id), из которых состоял заказ.

В качестве прогноза необходимо для каждой пары пользователь-категория из примера сабмита вернуть 1, если категория будет присутствовать в следующем заказе пользователя, или 0 в ином случае. Список категорий для каждого пользователя примере сабмита - это все категории, которые он когда-либо заказывал.

sample_submission.csv:

Пример сабмита. В тест входят не все пользователи из тренировочных данных, так как некоторые из них так ничего и не заказали после даты отсечки.

id - идентификатор строки - состоит из user_id и category_id, разделенных точкой с запятой: f'{user_id};{category_id}'. Из-за особенностей проверяющей системы Kaggle InClass, использовать колонки user_id, category_id в качестве индекса отдельно невозможно
target - 1 или 0 - будет ли данная категория присутствовать в следующем заказе пользователя


In [3]:
import pandas as pd
import os

In [4]:
# Блок: Загрузка и первичный анализ train.csv

train = pd.read_csv("train.csv")

print("Структура и информация о train.csv:")
print(train.info())

Структура и информация о train.csv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123064 entries, 0 to 3123063
Data columns (total 3 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   user_id             int64 
 1   order_completed_at  object
 2   cart                int64 
dtypes: int64(2), object(1)
memory usage: 71.5+ MB
None


In [5]:
print(f"Уникальных пользователей: {train['user_id'].nunique()}")

Уникальных пользователей: 20000


In [6]:
train

,user_id,order_completed_at,cart
0,2,2015-03-22 09:25:46,399
1,2,2015-03-22 09:25:46,14
2,2,2015-03-22 09:25:46,198
3,2,2015-03-22 09:25:46,88
4,2,2015-03-22 09:25:46,157
...,...,...,...
3123059,12702,2020-09-03 23:45:45,441
3123060,12702,2020-09-03 23:45:45,92
3123061,12702,2020-09-03 23:45:45,431
3123062,12702,2020-09-03 23:45:45,24


In [7]:
# Блок: Дополнительная статистика по train.csv

print("\nДополнительная статистика по train.csv:")

orders_per_user = train['user_id'].value_counts()
print("\nРаспределение количества заказов на пользователя:")
print(orders_per_user.describe())

print(f"\nУникальных категорий (корзин): {train['cart'].nunique()}")

train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])
print("\nСтатистика по датам заказов:")
print(f"Период с {train['order_completed_at'].min()} по {train['order_completed_at'].max()}")



Дополнительная статистика по train.csv:

Распределение количества заказов на пользователя:
count    20000.000000
mean       156.153200
std        200.840781
min          3.000000
25%         48.000000
50%         88.000000
75%        181.000000
max       3508.000000
Name: count, dtype: float64

Уникальных категорий (корзин): 881



Статистика по датам заказов:
Период с 2015-03-22 09:25:46 по 2020-09-03 23:45:45


In [8]:
# Блок: Загрузка и первичный анализ sample_submission.csv
sub = pd.read_csv('sample_submission.csv')

print("Структура и информация:")
print(sub.info())

Структура и информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790449 entries, 0 to 790448
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   id      790449 non-null  object
 1   target  790449 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 12.1+ MB
None


In [9]:
sub

,id,target
0,0;133,0
1,0;5,1
2,0;10,0
3,0;396,1
4,0;14,0
...,...,...
790444,19998;26,0
790445,19998;31,0
790446,19998;29,1
790447,19998;798,1


In [10]:
# Подробный анализ месяцев в трейне
import pandas as pd

# Убедимся, что дата уже преобразована
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])

# Создадим столбцы для года и месяца
train['year'] = train['order_completed_at'].dt.year
train['month'] = train['order_completed_at'].dt.month
train['year_month'] = train['order_completed_at'].dt.to_period('M')

print("Диапазон дат в тренировочных данных:")
print(f"От: {train['order_completed_at'].min().date()}")
print(f"До: {train['order_completed_at'].max().date()}")
print()

print("Распределение количества заказов по годам:")
print(train['year'].value_counts().sort_index())
print()

print("Количество уникальных месяцев (включая неполные):")
print(train['year_month'].nunique())
print()

Диапазон дат в тренировочных данных:
От: 2015-03-22
До: 2020-09-03

Распределение количества заказов по годам:
year
2015         52
2016       1375
2017       8938
2018      25464
2019     603398
2020    2483837
Name: count, dtype: int64

Количество уникальных месяцев (включая неполные):
60



In [11]:
# Извлекаем уникальные ID пользователей из сабмита
sub_user_ids = sub['id'].apply(lambda x: int(x.split(';')[0])).unique()

# Находим пересечение с пользователями из трейна
unique_train_users = set(train['user_id'].unique())
sub_users_in_train = [uid for uid in sub_user_ids if uid in unique_train_users]

print(f"Уникальных юзеров в сабмите: {len(sub_user_ids)}")
print(f"Уникальных юзеров из сабмита, которые есть в трейне: {len(sub_users_in_train)}")

# Процент покрытия
coverage_percentage = len(sub_users_in_train) / len(sub_user_ids) * 100
print(f"Процент юзеров из сабмита, присутствующих в трейне: {coverage_percentage:.2f}%")

Уникальных юзеров в сабмите: 13036
Уникальных юзеров из сабмита, которые есть в трейне: 13036
Процент юзеров из сабмита, присутствующих в трейне: 100.00%


In [12]:
# Извлекаем уникальные ID пользователей из сабмита
sub_user_ids = set(sub['id'].apply(lambda x: int(x.split(';')[0])).unique())

# Фильтруем трейн по 2020 году
train_2020 = train[train['year'] == 2020]
unique_users_2020 = set(train_2020['user_id'].unique())

# Проверяем пересечение
sub_users_in_2020 = sub_user_ids.intersection(unique_users_2020)

print(f"Всего уникальных юзеров в сабмите: {len(sub_user_ids)}")
print(f"Уникальных юзеров из сабмита, которые есть в 2020 году трейна: {len(sub_users_in_2020)}")
print(f"Процент покрытия: {len(sub_users_in_2020)/len(sub_user_ids)*100:.2f}%")

# Проверим, есть ли юзеры из сабмита, которых нет в 2020 году
missing_users = sub_user_ids - sub_users_in_2020
print(f"Юзеров из сабмита, которых НЕТ в 2020 году: {len(missing_users)}")

if len(missing_users) > 0:
    print("Несколько примеров таких юзеров:")
    print(list(missing_users)[:10])

Всего уникальных юзеров в сабмите: 13036
Уникальных юзеров из сабмита, которые есть в 2020 году трейна: 13036
Процент покрытия: 100.00%
Юзеров из сабмита, которых НЕТ в 2020 году: 0


In [13]:
# Извлекаем уникальные ID пользователей из сабмита
sub_user_ids = set(sub['id'].apply(lambda x: int(x.split(';')[0])).unique())

# Фильтруем трейн по 2020 году и пользователям из сабмита
train_2020 = train[(train['year'] == 2020) & (train['user_id'].isin(sub_user_ids))].copy()

# Создаем столбец с периодом месяц-год
train_2020['year_month'] = train_2020['order_completed_at'].dt.to_period('M')

# Подсчитываем количество уникальных пользователей из сабмита в каждом месяце 2020 года
users_per_month_2020 = train_2020.groupby('year_month')['user_id'].nunique()

print("Количество уникальных юзеров из сабмита, которые есть в каждом месяце 2020 года:")
for month, count in users_per_month_2020.items():
    print(f"{month}: {count}")

print(f"\nОбщее количество месяцев в 2020 году с пользователями из сабмита: {len(users_per_month_2020)}")

Количество уникальных юзеров из сабмита, которые есть в каждом месяце 2020 года:
2020-01: 2915
2020-02: 3194
2020-03: 4119
2020-04: 5386
2020-05: 6865
2020-06: 8795
2020-07: 10093
2020-08: 10546
2020-09: 2847

Общее количество месяцев в 2020 году с пользователями из сабмита: 9


In [14]:
# Фильтруем трейн по 2020 году и создаем столбец с месяцем
train_2020 = train[train['year'] == 2020].copy()
train_2020['month'] = train_2020['order_completed_at'].dt.month

# Находим уникальных пользователей, которые что-то купили в августе (месяц 8)
august_users = set(train_2020[train_2020['month'] == 8]['user_id'].unique())

# Находим уникальных пользователей, которые что-то купили в сентябре (месяц 9)
september_users = set(train_2020[train_2020['month'] == 9]['user_id'].unique())

print(f"Уникальных юзеров из трейна, купивших что-то в августе (месяц 8): {len(august_users)}")
print(f"Уникальных юзеров из трейна, купивших что-то в сентябре (месяц 9): {len(september_users)}")

# Находим пользователей, которые купили что-то в обоих месяцах
both_months_users = august_users.intersection(september_users)
print(f"Уникальных юзеров из трейна, купивших что-то и в августе, и в сентябре: {len(both_months_users)}")

# Общее количество уникальных пользователей, купивших что-то в августе или сентябре
either_month_users = august_users.union(september_users)
print(f"Уникальных юзеров из трейна, купивших что-то в августе или сентябре: {len(either_month_users)}")

Уникальных юзеров из трейна, купивших что-то в августе (месяц 8): 13353
Уникальных юзеров из трейна, купивших что-то в сентябре (месяц 9): 3169
Уникальных юзеров из трейна, купивших что-то и в августе, и в сентябре: 2723
Уникальных юзеров из трейна, купивших что-то в августе или сентябре: 13799


In [15]:
# Извлекаем уникальные ID пользователей из сабмита
sub_user_ids = set(sub['id'].apply(lambda x: int(x.split(';')[0])).unique())

# Фильтруем трейн по 2020 году, месяцам 8 и 9, и пользователям из сабмита
train_2020_aug_sep = train[(train['year'] == 2020) & 
                           (train['month'].isin([8, 9])) & 
                           (train['user_id'].isin(sub_user_ids))].copy()

# Создаем столбцы с месяцем для каждого заказа
train_2020_aug_sep['month'] = train_2020_aug_sep['order_completed_at'].dt.month

# Разделяем пользователей по месяцам
august_users_from_sub = set(train_2020_aug_sep[train_2020_aug_sep['month'] == 8]['user_id'].unique())
september_users_from_sub = set(train_2020_aug_sep[train_2020_aug_sep['month'] == 9]['user_id'].unique())

# Пользователи, которые купили что-то и в августе, и в сентябре (из сабмита)
both_months_users_from_sub = august_users_from_sub.intersection(september_users_from_sub)

print(f"Уникальных юзеров из сабмита, купивших что-то в августе (месяц 8): {len(august_users_from_sub)}")
print(f"Уникальных юзеров из сабмита, купивших что-то в сентябре (месяц 9): {len(september_users_from_sub)}")
print(f"Уникальных юзеров из сабмита, купивших что-то и в августе, и в сентябре: {len(both_months_users_from_sub)}")

# Общее количество уникальных пользователей из сабмита, купивших что-то в августе или сентябре
either_month_users_from_sub = august_users_from_sub.union(september_users_from_sub)
print(f"Уникальных юзеров из сабмита, купивших что-то в августе или сентябре: {len(either_month_users_from_sub)}")

Уникальных юзеров из сабмита, купивших что-то в августе (месяц 8): 10546
Уникальных юзеров из сабмита, купивших что-то в сентябре (месяц 9): 2847
Уникальных юзеров из сабмита, купивших что-то и в августе, и в сентябре: 2555
Уникальных юзеров из сабмита, купивших что-то в августе или сентябре: 10838


In [16]:
# Извлекаем уникальные ID пользователей из сабмита
sub_user_ids = set(sub['id'].apply(lambda x: int(x.split(';')[0])).unique())

# Фильтруем трейн по 2020 году и пользователям из сабмита
train_2020_sub = train[(train['year'] == 2020) & (train['user_id'].isin(sub_user_ids))].copy()

# Создаем столбцы с месяцем для каждого заказа
train_2020_sub['month'] = train_2020_sub['order_completed_at'].dt.month

# Находим пользователей, которые купили что-то в августе (месяц 8) или сентябре (месяц 9)
august_users = set(train_2020_sub[train_2020_sub['month'] == 8]['user_id'].unique())
september_users = set(train_2020_sub[train_2020_sub['month'] == 9]['user_id'].unique())
active_aug_or_sep = august_users.union(september_users)

# Находим пользователей из сабмита, которые НЕ были активны в августе или сентябре 2020
inactive_aug_sep = sub_user_ids - active_aug_or_sep

print(f"Всего юзеров из сабмита: {len(sub_user_ids)}")
print(f"Юзеров из сабмита, активных в авг или сен 2020: {len(active_aug_or_sep)}")
print(f"Юзеров из сабмита, НЕ активных в авг или сен 2020: {len(inactive_aug_sep)}")
print()

if len(inactive_aug_sep) > 0:
    # Фильтруем трейн по 2020 году и этим неактивным пользователям
    train_2020_inactive = train[(train['year'] == 2020) & (train['user_id'].isin(inactive_aug_sep))].copy()
    
    if len(train_2020_inactive) > 0:
        train_2020_inactive['month'] = train_2020_inactive['order_completed_at'].dt.month
        
        # Подсчитываем количество уникальных пользователей из этой группы в каждом месяце
        activity_by_month = train_2020_inactive.groupby('month')['user_id'].nunique()
        
        print("Количество активных юзеров из сабмита (НЕ активных в авг или сен), по месяцам 2020:")
        for month, count in activity_by_month.items():
            print(f"Месяц {month}: {count} активных юзеров")
        
        print(f"\\nМесяцы, когда эти юзеры были активны в 2020 году: {sorted(activity_by_month.index.tolist())}")
    else:
        print("Эти юзеры не были активны НИКОГДА в 2020 году")
else:
    print("Все юзеры из сабмита были активны в августе или сентябре 2020")

Всего юзеров из сабмита: 13036
Юзеров из сабмита, активных в авг или сен 2020: 10838
Юзеров из сабмита, НЕ активных в авг или сен 2020: 2198

Количество активных юзеров из сабмита (НЕ активных в авг или сен), по месяцам 2020:
Месяц 1: 480 активных юзеров
Месяц 2: 505 активных юзеров
Месяц 3: 711 активных юзеров
Месяц 4: 941 активных юзеров
Месяц 5: 1197 активных юзеров
Месяц 6: 1665 активных юзеров
Месяц 7: 1620 активных юзеров
\nМесяцы, когда эти юзеры были активны в 2020 году: [1, 2, 3, 4, 5, 6, 7]


In [19]:
# Преобразуем дату в формат datetime, если еще не сделано
train['order_completed_at'] = pd.to_datetime(train['order_completed_at'])

# Создаем столбцы для года и месяца
train['year'] = train['order_completed_at'].dt.year
train['month'] = train['order_completed_at'].dt.month

# Фильтруем только 2020 год
train_2020 = train[train['year'] == 2020]

# Получаем всех уникальных пользователей, которые были активны в каждом месяце 2020 года
months_2020 = train_2020['month'].unique()
print(f"Месяцы, представленные в данных за 2020 год: {sorted(months_2020)}")

# Группируем по пользователю и месяцу, чтобы найти уникальных пользователей в каждом месяце
user_month_activity = train_2020.groupby(['user_id', 'month']).size().reset_index(name='counts')

# Находим пользователей, которые были активны во всех месяцах 2020 года
users_active_in_all_months = user_month_activity['user_id'].value_counts()
fully_active_users = users_active_in_all_months[users_active_in_all_months == len(months_2020)]

print(f"Количество месяцев в 2020 году: {len(months_2020)}")
print(f"Количество пользователей, активных во всех {len(months_2020)} месяцах 2020 года: {len(fully_active_users)}")

Месяцы, представленные в данных за 2020 год: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9)]
Количество месяцев в 2020 году: 9
Количество пользователей, активных во всех 9 месяцах 2020 года: 375


In [33]:
# Берем первого пользователя из списка тех, кто был активен во всех 9 месяцах 2020 года
selected_user = fully_active_users.index[0]
print(f"Выбран пользователь: {selected_user}")

# Фильтруем данные для этого пользователя в 2020 году и сортируем по дате
user_data_2020 = train_2020[train_2020['user_id'] == selected_user].copy()
user_data_2020 = user_data_2020.sort_values('order_completed_at').reset_index(drop=True)

# Добавляем порядковый номер заказа
user_data_2020['order_number'] = range(1, len(user_data_2020) + 1)

# Создаем столбец для накопленных уникальных категорий
cumulative_categories = []
current_unique_cats = set()

for idx, row in user_data_2020.iterrows():
    current_unique_cats.add(int(row['cart']))
    cumulative_categories.append(current_unique_cats.copy())

user_data_2020['cumulative_categories'] = cumulative_categories

# Выводим данные для нескольких разных месяцев
print(f"\nДанные пользователя {selected_user} за разные месяцы 2020 года с накопленными уникальными категориями:")
for month in [1, 2, 3]:  # Выбираем три разных месяца для демонстрации
    if month in user_data_2020['month'].values:
        print(f"\n--- Месяц {month} ---")
        month_data = user_data_2020[user_data_2020['month'] == month].head(3)  # Берем первые 3 строки для наглядности
        print(month_data[['order_number', 'user_id', 'order_completed_at', 'cart', 'month', 'cumulative_categories']].to_markdown(index=True))
        
        # Показываем итоговое количество уникальных категорий по итогам месяца
        max_order_in_month = user_data_2020[user_data_2020['month'] == month]['order_number'].max()
        final_cumulative_for_month = user_data_2020[user_data_2020['order_number'] == max_order_in_month]['cumulative_categories'].iloc[0]
        print(f"Итого уникальных категорий после месяца {month}: {len(final_cumulative_for_month)}")
    else:
        print(f"\n--- Месяц {month} ---")
        print("Нет данных для этого месяца")

Выбран пользователь: 186

Данные пользователя 186 за разные месяцы 2020 года с накопленными уникальными категориями:

--- Месяц 1 ---
|    |   order_number |   user_id | order_completed_at   |   cart |   month | cumulative_categories   |
|---:|---------------:|----------:|:---------------------|-------:|--------:|:------------------------|
|  0 |              1 |       186 | 2020-01-05 09:17:48  |    208 |       1 | {208}                   |
|  1 |              2 |       186 | 2020-01-05 09:17:48  |      6 |       1 | {208, 6}                |
|  2 |              3 |       186 | 2020-01-05 09:17:48  |     84 |       1 | {208, 84, 6}            |
Итого уникальных категорий после месяца 1: 46

--- Месяц 2 ---
|    |   order_number |   user_id | order_completed_at   |   cart |   month | cumulative_categories                                                                                                                                                                                        

In [46]:
# Создаем новый датафрейм на основе отсортированных данных пользователя
new_df = user_data_2020.copy()

# Добавляем столбец target
target_values = []

for idx, row in new_df.iterrows():
    current_month = row['month']
    
    if current_month == 1:  # Первый месяц
        target_values.append(1)
    else:
        # Находим максимальный номер заказа в предыдущем месяце
        prev_month = current_month - 1
        prev_month_data = new_df[(new_df['month'] == prev_month)]
        
        if not prev_month_data.empty:
            max_order_prev_month = prev_month_data['order_number'].max()
            # Находим накопленные категории на момент завершения предыдущего месяца
            cumulative_at_end_prev_month = new_df[new_df['order_number'] == max_order_prev_month]['cumulative_categories'].iloc[0]
            
            # Проверяем, был ли текущий cart в накопленных категориях предыдущего месяца
            if row['cart'] in cumulative_at_end_prev_month:
                target_values.append(1)
            else:
                target_values.append(0)
        else:
            # Если нет данных о предыдущем месяце, ставим 0
            target_values.append(0)

new_df['target'] = target_values

print("Распределение таргета в полном датафрейме (до фильтрации):")
for month in sorted(new_df['month'].unique()):
    month_data = new_df[new_df['month'] == month]
    if month == 1:
        target_counts = {1: len(month_data)}  # Все в первом месяце = 1
        print(f"Месяц {month}: {target_counts} (все строки = 1 по определению)")
    else:
        target_counts = month_data['target'].value_counts().to_dict()
        print(f"Месяц {month}: {target_counts}")
        
        # Покажем примеры строк с таргетом 0 (если есть)
        if 0 in target_counts:
            sample_target_0 = month_data[month_data['target'] == 0].head(3)
            print(f"  Примеры строк с таргет=0 (новые категории, не встречавшиеся ранее):")
            for idx, row in sample_target_0.iterrows():
                print(f"    Заказ {row['order_number']}, cart={row['cart']}, дата={row['order_completed_at'].strftime('%Y-%m-%d')}")
        
        # И примеры строк с таргетом 1
        sample_target_1 = month_data[month_data['target'] == 1].head(3)
        print(f"  Примеры строк с таргет=1 (категории, встречавшиеся ранее):")
        for idx, row in sample_target_1.iterrows():
            print(f"    Заказ {row['order_number']}, cart={row['cart']}, дата={row['order_completed_at'].strftime('%Y-%m-%d')}")

print(f"\nПолный датафрейм (с таргетом, без фильтрации) - это то, что содержит таргеты 0 и 1:")
print(f"Размер: {len(new_df)} строк")
target_stats_full = new_df['target'].value_counts().sort_index()
print(f"Распределение таргета в полном датафрейме: {dict(target_stats_full)}")

for target_val in sorted(target_stats_full.index):
    count = target_stats_full[target_val]
    percentage = count / len(new_df) * 100
    print(f"Таргет {target_val}: {count} строк ({percentage:.2f}%)")

Распределение таргета в полном датафрейме (до фильтрации):
Месяц 1: {1: 80} (все строки = 1 по определению)
Месяц 2: {1: 47, 0: 13}
  Примеры строк с таргет=0 (новые категории, не встречавшиеся ранее):
    Заказ 83, cart=438, дата=2020-02-01
    Заказ 93, cart=92, дата=2020-02-01
    Заказ 94, cart=440, дата=2020-02-01
  Примеры строк с таргет=1 (категории, встречавшиеся ранее):
    Заказ 81, cart=82, дата=2020-02-01
    Заказ 82, cart=14, дата=2020-02-01
    Заказ 84, cart=84, дата=2020-02-01
Месяц 3: {1: 60, 0: 22}
  Примеры строк с таргет=0 (новые категории, не встречавшиеся ранее):
    Заказ 142, cart=172, дата=2020-03-03
    Заказ 144, cart=403, дата=2020-03-03
    Заказ 149, cart=169, дата=2020-03-03
  Примеры строк с таргет=1 (категории, встречавшиеся ранее):
    Заказ 141, cart=55, дата=2020-03-03
    Заказ 143, cart=434, дата=2020-03-03
    Заказ 145, cart=57, дата=2020-03-03
Месяц 4: {1: 64, 0: 17}
  Примеры строк с таргет=0 (новые категории, не встречавшиеся ранее):
    Зака

In [48]:
print("Статистика по таргету в полном датафрейме (new_df) - до фильтрации:")
print(f"Размер датафрейма: {len(new_df)} строк")

target_stats = new_df['target'].value_counts().sort_index()
print(f"Распределение таргета: {dict(target_stats)}")

for target_val in sorted(target_stats.index):
    count = target_stats[target_val]
    percentage = count / len(new_df) * 100
    print(f"Таргет {target_val}: {count} строк ({percentage:.2f}%)")

print(f"\nРаспределение таргета по месяцам в полном датафрейме:")
for month in sorted(new_df['month'].unique()):
    month_data = new_df[new_df['month'] == month]
    month_target_dist = month_data['target'].value_counts().sort_index().to_dict()
    print(f"Месяц {month}: {month_target_dist}")

print(f"\nПримеры строк с таргетом 0 (если есть):")
target_0_examples = new_df[new_df['target'] == 0].head(5)
if not target_0_examples.empty:
    for idx, row in target_0_examples.iterrows():
        print(f"  Заказ {row['order_number']}, месяц={row['month']}, cart={row['cart']}, дата={row['order_completed_at'].strftime('%Y-%m-%d')}")
else:
    print("  Нет строк с таргетом 0")

Статистика по таргету в полном датафрейме (new_df) - до фильтрации:
Размер датафрейма: 559 строк
Распределение таргета: {0: np.int64(81), 1: np.int64(478)}
Таргет 0: 81 строк (14.49%)
Таргет 1: 478 строк (85.51%)

Распределение таргета по месяцам в полном датафрейме:
Месяц 1: {1: 80}
Месяц 2: {0: 13, 1: 47}
Месяц 3: {0: 22, 1: 60}
Месяц 4: {0: 17, 1: 64}
Месяц 5: {0: 12, 1: 70}
Месяц 6: {0: 9, 1: 51}
Месяц 7: {0: 4, 1: 53}
Месяц 8: {0: 4, 1: 44}
Месяц 9: {1: 9}

Примеры строк с таргетом 0 (если есть):
  Заказ 83, месяц=2, cart=438, дата=2020-02-01
  Заказ 93, месяц=2, cart=92, дата=2020-02-01
  Заказ 94, месяц=2, cart=440, дата=2020-02-01
  Заказ 101, месяц=2, cart=25, дата=2020-02-17
  Заказ 102, месяц=2, cart=54, дата=2020-02-17


In [49]:
print("Проверка строк с таргетом 0 и соответствующих накопленных категорий предыдущего месяца:")
print(f"Пользователь: {selected_user}")

# Возьмем несколько примеров строк с таргетом 0
target_0_examples = new_df[new_df['target'] == 0].head(5)

for idx, row in target_0_examples.iterrows():
    current_month = row['month']
    prev_month = current_month - 1
    
    print(f"\n--- Пример строки с таргетом 0 ---")
    print(f"Заказ {row['order_number']}, месяц={current_month}, cart={row['cart']}, дата={row['order_completed_at'].strftime('%Y-%m-%d')}")
    
    # Найдем накопленные категории на конец предыдущего месяца
    prev_month_data = new_df[(new_df['month'] == prev_month)]
    if not prev_month_data.empty:
        max_order_prev_month = prev_month_data['order_number'].max()
        cumulative_at_end_prev_month = new_df[new_df['order_number'] == max_order_prev_month]['cumulative_categories'].iloc[0]
        
        print(f"Максимальный номер заказа в предыдущем месяце ({prev_month}): {max_order_prev_month}")
        print(f"Накопленные категории на конец месяца {prev_month} ({len(cumulative_at_end_prev_month)} шт.): {cumulative_at_end_prev_month}")
        print(f"Cart {row['cart']} входит в накопленные категории предыдущего месяца: {row['cart'] in cumulative_at_end_prev_month}")
        print(f"Поскольку {row['cart']} НЕ входит в накопленные категории, таргет = 0")
    else:
        print(f"Нет данных о предыдущем месяце {prev_month}")

# Также посмотрим примеры строк с таргетом 1 для сравнения
print(f"\n--- Для сравнения: примеры строк с таргетом 1 ---")
target_1_examples = new_df[(new_df['target'] == 1) & (new_df['month'] > 1)].head(3)

for idx, row in target_1_examples.iterrows():
    current_month = row['month']
    prev_month = current_month - 1
    
    print(f"\nЗаказ {row['order_number']}, месяц={current_month}, cart={row['cart']}, дата={row['order_completed_at'].strftime('%Y-%m-%d')}")
    
    # Найдем накопленные категории на конец предыдущего месяца
    prev_month_data = new_df[(new_df['month'] == prev_month)]
    if not prev_month_data.empty:
        max_order_prev_month = prev_month_data['order_number'].max()
        cumulative_at_end_prev_month = new_df[new_df['order_number'] == max_order_prev_month]['cumulative_categories'].iloc[0]
        
        print(f"Накопленные категории на конец месяца {prev_month} ({len(cumulative_at_end_prev_month)} шт.): {cumulative_at_end_prev_month}")
        print(f"Cart {row['cart']} входит в накопленные категории предыдущего месяца: {row['cart'] in cumulative_at_end_prev_month}")
        print(f"Поскольку {row['cart']} ВХОДИТ в накопленные категории, таргет = 1")

Проверка строк с таргетом 0 и соответствующих накопленных категорий предыдущего месяца:
Пользователь: 186

--- Пример строки с таргетом 0 ---
Заказ 83, месяц=2, cart=438, дата=2020-02-01
Максимальный номер заказа в предыдущем месяце (1): 80
Накопленные категории на конец месяца 1 (46 шт.): {386, 6, 393, 396, 398, 14, 400, 17, 146, 402, 21, 24, 29, 798, 31, 417, 419, 420, 421, 805, 807, 808, 425, 170, 427, 430, 431, 179, 55, 57, 441, 443, 61, 198, 199, 204, 208, 82, 84, 85, 86, 220, 229, 236, 248, 383}
Cart 438 входит в накопленные категории предыдущего месяца: False
Поскольку 438 НЕ входит в накопленные категории, таргет = 0

--- Пример строки с таргетом 0 ---
Заказ 93, месяц=2, cart=92, дата=2020-02-01
Максимальный номер заказа в предыдущем месяце (1): 80
Накопленные категории на конец месяца 1 (46 шт.): {386, 6, 393, 396, 398, 14, 400, 17, 146, 402, 21, 24, 29, 798, 31, 417, 419, 420, 421, 805, 807, 808, 425, 170, 427, 430, 431, 179, 55, 57, 441, 443, 61, 198, 199, 204, 208, 82, 84, 